In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
sooyoungher_smoking_drinking_dataset_path = kagglehub.dataset_download('sooyoungher/smoking-drinking-dataset')

print('Data source import complete.')


<h1 style="font-family:'Gill Sans', sans-serif; color:#0297DC;"> <center> Smoking and Drinking Classification with Body Signal </center> </h1>
<center><img src="https://media.tenor.com/jC5ZTCxRKEUAAAAC/cheers-drinking.gif" width="500"/></center>

<strong><h3>Dataset Story</h2></strong>
The dataset is collected from the National Health Insurance Service in Korea. All personal information and sensitive data were excluded. The aim is to predict whether a person is a smoker or drinker according to body signals.

Let's check out the datasets, start with EDA with both the classical approach and advanced EDA library known as pandas-profiling, and finally create models with both the classical approach and PyCar


<li><strong>Total Features : 24 </strong></li>
<li><strong>Total Row : 991346 </strong> </li>
<li><strong>CSV File Size : 104.486 MB</strong></li>
<br>


<div class="inner_cell">
<div class="text_cell_render border-box-sizing rendered_html">
<p></p><div class="list-group" id="list-tab" role="tablist">
  <h3 class="list-group-item list-group-item-action active" data-toggle="list" role="tab" aria-controls="home" style = "border:2px solid #87CEFA;background-color:#87CEFA; color:white; font-family:Verdana;text-align: center; font-size:140%;font-weight: Bold;">Notebook Content</h3>
  <a class="list-group-item list-group-item-action" data-toggle="list" href="#1" role="tab" aria-controls="profile" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Import Libraries and Check Data<span class="badge badge-primary badge-pill">1</span></a>
  <a class="list-group-item list-group-item-action" data-toggle="list" href="#2" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Exploratory Data Analysis(EDA) with Classical Approach<span class="badge badge-primary badge-pill">2</span></a>
  <a class="list-group-item list-group-item-action" data-toggle="list" href="#3" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Exploratory Data Analysis(EDA) with pandas-profiling<span class="badge badge-primary badge-pill">3</span></a>
  <a class="list-group-item list-group-item-action" data-toggle="list" href="#4" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Base Model with Classical Approach<span class="badge badge-primary badge-pill">4</span></a>
 <a class="list-group-item list-group-item-action" data-toggle="list" href="#5" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Base Model with Lazy Predict & PyCaret<span class="badge badge-primary badge-pill">5</span></a>
  <a class="list-group-item list-group-item-action" data-toggle="list" href="#6" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Feature Engineering<span class="badge badge-primary badge-pill">6</span></a>
 <a class="list-group-item list-group-item-action" data-toggle="list" href="#7" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Hyperparameter Optimization with Optuna<span class="badge badge-primary badge-pill">7</span></a>
 <a class="list-group-item list-group-item-action" data-toggle="list" href="#8" role="tab" aria-controls="messages" target="_self" style = "color:##87CEFA; font-family:Verdana;text-align: center; font-size:130%;font-weight: Bold;">Final Model<span class="badge badge-primary badge-pill">8</span></a>

    
</div>
</div>
</div>

---

<a id="1"></a> <br>
# 1. Import Libraries and Check Data 🔎

In [ ]:
# Basic Libraries 📚
# -------------------
import numpy as np
import pandas as pd

# Visualization Libraries 📊
# ------------------------------
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno

# Machine Learning Models 🤖
# --------------------------------------------------------------------------------------------------
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score, precision_score


# Customize to Remove Warnings and Better Observation 🔧
# --------------------------------------------------------
from termcolor import colored
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

Since the dataset is slightly large, we will convert CSV to parquet for much more speed and less memory size. For more information about parquet, you can check these notebooks:

**[Lightweight Data (~20MB)](https://www.kaggle.com/code/furkannakdagg/lightweight-data-20mb)**

**[Parquet Spotify Dataset(~x3.3 faster & x2 lighter)](https://www.kaggle.com/code/furkannakdagg/parquet-spotify-dataset-x3-3-faster-x2-lighter)**

In [ ]:
# Convert csv to parquet
pd.read_csv("/kaggle/input/smoking-drinking-dataset/smoking_driking_dataset_Ver01.csv").to_parquet("smoking_driking_dataset.parquet")

Now, check the file size and speeds to compare CSV and parquet.

In [ ]:
# For finding file size
import os
def print_file_size(file_path):
    file_stats = os.stat(file_path)
    file_size = np.round(file_stats.st_size / (1024 * 1024), 3)
    # print(f'File Size: {file_size} MB')
    return file_size

csv_file_size = print_file_size("/kaggle/input/smoking-drinking-dataset/smoking_driking_dataset_Ver01.csv")
print(f"CSV File Size: {csv_file_size} MB")
parq_file_size = print_file_size("smoking_driking_dataset.parquet")
print(f"Parquet File Size: {parq_file_size} MB")
print(f"Parques is {round(csv_file_size/parq_file_size,2)}x lighter!")

In [ ]:
df = pd.read_parquet("smoking_driking_dataset.parquet")
df.head()

In [ ]:
df.columns = df.columns.map(str.lower)
# df.columns = [col.lower() for col in df.columns]
df.columns

In [ ]:
df["drk_yn"] = np.where(df["drk_yn"] == "Y", 1, 0)
df.head()

In [ ]:
TARGET = "drk_yn"

<a id="2"></a> <br>
# 2. Exploratory Data Analysis(EDA) with Classical Approach 🧐

First, we will look at the classical approach with functions. In section 3, we will complete the EDA with pandas-profiling.

In [ ]:
def check_df(dataframe, head=5):
    print("##################### Shape #####################")
    print(dataframe.shape)
    print("##################### Types #####################")
    print(dataframe.dtypes)
    print("##################### Head #####################")
    display(dataframe.head(head))
    print("##################### Tail #####################")
    display(dataframe.tail(head))
    print("##################### NA #####################")
    print(dataframe.isnull().sum())
    print("##################### Quantiles #####################")
    display(dataframe.quantile([0, 0.05, 0.50, 0.95, 0.99, 1]).T)

check_df(df)

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    """
    It gives the names of categorical, numerical, and categorical but cardinal variables in the data set.
    Note: Categorical variables with numerical appearance are also included in categorical variables.

    Parameters
    ------
        dataframe: dataframe
                dataframe
        cat_th: int, optional
                threshold value for variables that appear numeric but are categorical
        car_th: int, optional
                threshold value for categorical but cardinal variables

    Returns
    ------
        cat_cols: list
                Categorical variable list
        num_cols: list
                Numerical variable list
        cat_but_car: list
                Cardinal variable list

    Examples
    ------
        import seaborn as sns
        df = sns.load_dataset("iris")
        print(grab_col_names(df))


    Notes
    ------
        cat_cols + num_cols + cat_but_car = total number of variables
        num_but_cat is inside cat_cols

    """
    # cat_cols, cat_but_car
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]
    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and dataframe[col].dtypes != "O"]
    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and dataframe[col].dtypes == "O"]
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # num_cols
    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]
    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')

    return cat_cols, num_cols, cat_but_car


cat_cols, num_cols, cat_but_car = grab_col_names(df)
print(f"\n{colored('Numerical Columns:','blue', attrs=['reverse'])} {num_cols}\n\n\n{colored('Categorical Columns:','magenta', attrs=['reverse'])} {cat_cols}\n\n\n"
        f"{colored('Cardinal Columns:','cyan', attrs=['reverse'])}{cat_but_car}\n")

In [ ]:
###################################################################
# 2. Analysis of Categorical Variables (Kategorik Değişken Analizi)
###################################################################
def cat_summary(dataframe, col_name, plot=False):
    display(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        "Ratio": 100 * dataframe[col_name].value_counts() / len(dataframe)}))

    if plot:
        sns.countplot(x=dataframe[col_name], data=dataframe)
        plt.show()


for col in cat_cols:
    cat_summary(df, col)
    print("")

In [ ]:
###############################################################
# 3. Analysis of Numerical Variables (Sayısal Değişken Analizi)
###############################################################
def num_summary(dataframe, numerical_col, plot=False):
    quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    print(dataframe[numerical_col].describe(quantiles).T)

    if plot:
        dataframe[numerical_col].hist(bins=50)
        plt.xlabel(numerical_col)
        plt.title(numerical_col)
        plt.show(block=True)

    print("#####################################")


for col in num_cols:
    num_summary(df, col, True)

In [ ]:
#####################################################################################################################
# 3. Analysis of Categorical Variables on the Target Variable (Kategorik Değişkenlerin Hedef Değişkene Göre Analizi)
####################################################################################################################

def target_summary_with_cat(dataframe, target, categorical_col):
    print(pd.DataFrame({"TARGET_MEAN": dataframe.groupby(categorical_col)[target].mean()}), end="\n\n\n")


for col in cat_cols:
    target_summary_with_cat(df,TARGET,col)

In [ ]:
################################################################################################################
# 4. Analysis of Numerical Variables on the Target Variable (Numerik Değişkenlerin Hedef Değişkene Göre Analizi)
################################################################################################################

def target_summary_with_num(dataframe, target, numerical_col):
    print(dataframe.groupby(target).agg({numerical_col: "mean"}), end="\n\n\n")

for col in num_cols:
    target_summary_with_num(df, TARGET, col)

In [ ]:
##############################################
# 5. Correlation Analysis (Korelasyon Analizi)
##############################################

def corr_map(df, width=14, height=6, annot_kws=15):
    mtx = np.triu(df.corr())
    f, ax = plt.subplots(figsize = (width,height))
    sns.heatmap(df.corr(),
                annot= True,
                fmt = ".2f",
                ax=ax,
                vmin = -1,
                vmax = 1,
                cmap = "RdBu",
                mask = mtx,
                linewidth = 0.4,
                linecolor = "black",
                cbar=False,
                annot_kws={"size": annot_kws})
    plt.yticks(rotation=0,size=15)
    plt.xticks(rotation=75,size=15)
    plt.title('\nCorrelation Map\n', size = 20)
    plt.show();

corr_map(df, width=20, height=10, annot_kws=8)

We observed that there is no missing value on the data frame, nevertheless, let's check out the function.

In [ ]:
################################################
# 6. Missing Valy Analysis (Eksik Değer Analizi)
################################################

def missing_values_table(dataframe, na_name=False):
    na_columns = [col for col in dataframe.columns if dataframe[col].isnull().sum() > 0]
    n_miss = dataframe[na_columns].isnull().sum().sort_values(ascending=False)
    ratio = (dataframe[na_columns].isnull().sum() / dataframe.shape[0] * 100).sort_values(ascending=False)
    missing_df = pd.concat([n_miss, np.round(ratio, 2)], axis=1, keys=['n_miss', 'ratio'])
    print(missing_df, end="\n")
    if na_name:
        return na_columns

missing_values_table(df)

In [ ]:
####################################################################################
# 7. Advanced Plots for Missing Value Analysis (Eksik Değer Analizi Gelişmiş Grafikler)
####################################################################################
def msno_plots(dataframe):
    msno.bar(dataframe)
    plt.title("Nullity Bar", fontsize=25)
    plt.show()

    print("".center(100, "~"))

    msno.matrix(dataframe)
    plt.title("Nullity Matrix", fontsize=25)
    plt.show()

    print("".center(100, "~"))

    msno.heatmap(dataframe)
    plt.title("Nullity Correlation", fontsize=25)
    plt.show()

msno_plots(df)

In [ ]:
############################
# 8. BONUS - Duplicated Rows
############################
def duplicated_rows(dataframe, head=5, report=True, dup_idx=False, drop_dup=False):
    if report:
        df_dup = df[df.duplicated()]
        print(f"Number of Duplicated Rows: {df_dup.shape[0]}\n")
        print(f"Duplicated first {head} Rows:")
        display(df_dup.head(head))

    if dup_idx:
        return df_dup.index

    if drop_dup:
        dataframe = dataframe.drop(df_dup.index, axis=0)
        return dataframe

duplicated_rows(df)

<a id="3"></a> <br>
# 3. Exploratory Data Analysis(EDA) with pandas-profiling 🐼

pandas-profiling, a.k.a. ydata-profiling, whose primary goal is to provide a one-line Exploratory Data Analysis (EDA) experience in a consistent and fast solution. Like pandas df.describe() function, which is so handy, ydata-profiling delivers an extended analysis of a DataFrame while allowing the data analysis to be exported in different formats such as HTML and JSON.

For detailed information, check the documentation:
> **[pandas-profiling Documentation](https://ydata-profiling.ydata.ai/docs/master/)**

In [ ]:
!pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="Pandas Profiling Report")
profile
# profile.to_notebook_iframe()

<div class="alert alert-success alert-info">
       <b> 📌 With this library, we can check the detailed analysis and explore the data easily in a few minutes. The report generation time will vary depending on the data set size. </b>
</div>
<br>

---

### pandas-profiling Features
1. We can use the minimal=True argument when working with large data sets. So we can quickly review the summary statistics.

In [ ]:
ProfileReport(df, minimal=True)

We don't need to execute pandas-profiling every time. By saving the report, we can re-examine the report at any time without running the codes.

In [ ]:
# For saving report
profile.to_file("my_report.html")  # the report will appear on Output section

2. It also allows us to compare data sets. Where we have train and test datasets, we can also compare these datasets. In this way, we can observe whether our test set can successfully express our train set.

> Since we do not have separate train-tests here, we create our own train and test sets by sampling from df to be an example. We will take small samples to make the running time as short as possible. Also, minimal=True will be used for a faster process.

In [ ]:
from pandas_profiling import ProfileReport, compare
train = df.sample(100, random_state=42)
test = df.sample(10, random_state=42)

In [ ]:
train_report = ProfileReport(train, title="Train", minimal=True)
test_report = ProfileReport(test, title="Test", minimal=True)
comparison_report = train_report.compare(test_report)
comparison_report

<a id="4"></a> <br>
# 4. Base Model with Classical Approach 🧱

We will save our df to another data frame named dff because we may want to save the original format of our main data frame. The process will continue with dff.

Also, since we have 1 million rows, CV processes take a long time. As an example of show the process and libraries, we will take 10k random data to set up the models quickly because 1m rows will significantly increase the processing time and Kaggle may not handle this situation. You can try it yourself whenever you have time.

In [ ]:
dff = df.sample(10000, random_state=42).reset_index(drop=True).copy()

print(dff.shape)

# re-orginize cat_cols since we will make some process on these
cat_cols = [col for col in cat_cols if col not in [TARGET]]
print(cat_cols)

We got 10k samples but we have to be sure whether this sample can represent our main dataset. There are different ways to control this situation. For now, we will quickly control their descriptive statistics. Remember, pandas-profiling helps us to compare data sets!

In [ ]:
from pandas_profiling import ProfileReport, compare
df_report = ProfileReport(df, title="Original Dataset", minimal=True)
dff_report = ProfileReport(dff, title="Sample", minimal=True)
comparison_report = df_report.compare(dff_report)
comparison_report

<div class="alert alert-block alert-info">
        <b> 📌 Since the descriptive statistics seem similar, we can move with the sample data set.</b>
</div>
<br>

Normally we had to encode our categorical variables, but since they are all numeric here, we can skip this step. For now, we will only apply Label Encoder to the "sex" variable. If our model performance in the future is not as we expected, we can go back and apply this step and build the model again.

In [ ]:
def one_hot_encoder(dataframe, categorical_cols, drop_first=False):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe


def label_encoder(dataframe, binary_col):
    labelencoder = LabelEncoder()
    dataframe[binary_col] = labelencoder.fit_transform(dataframe[binary_col])
    return dataframe

for col in ["sex"]:
    dff = label_encoder(dff, col)

In [ ]:
y = dff[TARGET]
X = dff.drop(TARGET, axis=1)

We can proceed with CV as model validation, but if our dataset is too large, this prolongs the processing time. In such a case, we may continue with the hold-out method. Both CV and Hold-out methods are given below, we will continue with CV.

In [ ]:
models = [('LR', LogisticRegression(random_state=12345)),
          ('KNN', KNeighborsClassifier()),
          ('CART', DecisionTreeClassifier(random_state=12345)),
          ('RF', RandomForestClassifier(random_state=12345)),
          ('XGB', XGBClassifier(random_state=12345)),
          ("LightGBM", LGBMClassifier(device='gpu', random_state=12345)),
          # ("CatBoost", CatBoostClassifier(task_type="GPU", verbose=False, random_state=12345))
         ]


for name, model in models:
    cv_results = cross_validate(model, X, y, cv=5, scoring=["accuracy", "f1", "roc_auc", "precision", "recall"])
    print(f"########## {name} ##########")
    print(f"Accuracy: {round(cv_results['test_accuracy'].mean(), 4)}")
    print(f"Auc: {round(cv_results['test_roc_auc'].mean(), 4)}")
    print(f"Recall: {round(cv_results['test_recall'].mean(), 4)}")
    print(f"Precision: {round(cv_results['test_precision'].mean(), 4)}")
    print(f"F1: {round(cv_results['test_f1'].mean(), 4)}")

In [ ]:
# You can use hold-out, as well.
"""
models = [('LR', LogisticRegression(random_state=12345)),
          # ('KNN', KNeighborsClassifier(n_jobs=-1)),
          ('CART', DecisionTreeClassifier(random_state=12345)),
          ('RF', RandomForestClassifier(random_state=12345)),
          ('XGB', XGBClassifier(random_state=12345)),
          ("LightGBM", LGBMClassifier(device='gpu', random_state=12345)),
          ("CatBoost", CatBoostClassifier(task_type="GPU", verbose=False, random_state=12345))
         ]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=17)
for name, model in models:
    base_model = model.fit(X_train, y_train)
    y_pred = base_model.predict(X_test)
    print(f"########## {name} ##########")
    print(f"Accuracy: {round(accuracy_score(y_pred, y_test), 4)}")
    print(f"Auc: {round(roc_auc_score(y_pred, y_test), 4)}")
    print(f"Recall: {round(recall_score(y_pred, y_test), 4)}")
    print(f"Precision: {round(precision_score(y_pred, y_test), 4)}")
    print(f"F1: {round(f1_score(y_pred, y_test), 4)}")
"""

<a id="5"></a> <br>
# 5. Base Model with Lazy Predict & PyCaret 💎

## 5.1) LazyPredict

Lazy Predict helps build a lot of basic models without much code and helps understand which models works better without any parameter tuning. The library reflect its name as "lazy". It has no complicated arguments, it simply run the models and show the scores, there is no choice to get best model in this library.

> **Documentation: [Lazy Predict Documentation](https://lazypredict.readthedocs.io/en/latest/)**

In [ ]:
!pip install lazypredict

Lazy Predict gets X_train, X_test, y_train, and y_test on .fit() which is different from the regular way of the creating model.

In [ ]:
from lazypredict.Supervised import LazyClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42)
reg = LazyClassifier(verbose=-1, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [ ]:
models

If there is a custom metric we want to use in order to sort models according to model performance, we can use this custom metric with the <ins>custom_metric</ins> argument.

In my observation, giving a custom metric from sklearn is much more reliable.

In [ ]:
reg = LazyClassifier(verbose=-1, ignore_warnings=True, custom_metric=f1_score)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

In [ ]:
models

## 5.2 PyCaret

PyCaret is an open-source, low-code machine learning library in Python that automates machine learning workflows. It is an end-to-end machine learning and model management tool that exponentially speeds up the experiment cycle and makes you more productive. This is one of the amazing libraries among Auto-ML tools, I strongly recommend you check the documentation.

Documentation: [PyCaret Documentation](https://pycaret.gitbook.io/docs/)

In [ ]:
!pip install pycaret

Many features can be added in PyCaret, such as target variable, categorical and numeric variables, cardinal variables to be ignored. For features that can be added in the setup for classification problems:

[Classification Setup](https://pycaret.readthedocs.io/en/latest/api/classification.html)

In [ ]:
# fold=3 for shorter run time, default fold=10
from pycaret.classification import *
s = setup(dff, target = TARGET, session_id = 123,
          train_size=0.8, categorical_features=cat_cols, fold=3, use_gpu=True)

PyCaret gives a summary of how the models will be created. It is so functioning as you can see. Not only folding, and categorical features, but also we even can specify how the numerical and categorical missing values will be imputed.

In [ ]:
best = compare_models()

In [ ]:
best

PyCaret compares all models and saves the model with the best score. The default scoring is "Accuracy" when models are compared. We can replace this with the <ins>sort</ins> argument.

> **best = compare_models(sort="F1")**

In [ ]:
# Creating the model by selecting the best model found
model = create_model(best)

In [ ]:
# Making prediction with model
best_model_pred = predict_model(model)

In [ ]:
best_model_pred

In [ ]:
# Getting predictions labels
best_model_pred["prediction_label"]

Regardless of the scores we receive, we can also build a model with the specific algorithm we want. If we want to build a model with the algorithm we choose:

In [ ]:
lgbm = create_model('lightgbm')

One of the features of PyCaret is hyperparameter tuning. We can do this with <ins>tune_model</ins> function.

In [ ]:
tuned_lgbm = tune_model(lgbm)

Another feature is that it offers us plenty of visualization opportunities. We can also visualize our model with <ins>plot_model</ins> function.

In [ ]:
plot_model(tuned_lgbm, plot = 'auc')

In [ ]:
plot_model(tuned_lgbm, plot = 'feature')

In [ ]:
plot_model(tuned_lgbm, plot = 'confusion_matrix')

We can also give our own parameter set while hyperparameter tuning. We chose fold and iteration as 3 to complete the process in a short time.

In [ ]:
lgbm_params = {
    "max_depth": np.arange(2, 10),
    "learning_rate": [round(i,2) for i in np.linspace(0.0001, 0.2, num=100)],
    "colsample_bytree": [round(i,2) for i in np.linspace(0.5, 1, num=3)],
    "colsample_bynode": [round(i,2) for i in np.linspace(0.5, 1, num=3)],
    "num_leaves": np.arange(10, 100)
}

tuned_lgbm = tune_model(lgbm, custom_grid=lgbm_params, n_iter=3, fold=3)

After completing our model, we can give the data set we want to predict as an argument and extract the prediction scores from it.

In [ ]:
preds = predict_model(tuned_lgbm, data = dff)

In [ ]:
preds

In [ ]:
preds["prediction_label"]

<a id="6"></a> <br>
# 6. Feature Engineering 🏗

We will remove outliers with IQR method.

In [ ]:
#################################
#  Outliers (Aykırı Değer Analizi)
#################################

# Find outlier thresholds
def outlier_thresholds(dataframe, col_name, q1=0.05, q3=0.95):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit


def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

In [ ]:
for col in num_cols:
    print(col, check_outlier(dff, col))

In [ ]:
def replace_with_thresholds(dataframe, variable, q1=0.05, q3=0.95):
    low_limit, up_limit = outlier_thresholds(dataframe, variable, q1=0.05, q3=0.95)
    dataframe.loc[(dataframe[variable] < low_limit), variable] = low_limit
    dataframe.loc[(dataframe[variable] > up_limit), variable] = up_limit

for col in num_cols:
    replace_with_thresholds(dff, col)

After replacing outliers, let's check again for outliers.

In [ ]:
for col in num_cols:
    print(col, check_outlier(dff, col))

In [ ]:
bins = [18, 30, 45, df.age.max()]
labels = ['young', 'middle_age', 'old_age']
dff['AGE_CAT'] = pd.cut(dff['age'], bins=bins, labels=labels)
dff["HEMOGLOBIN_CAT"] = pd.qcut(dff["hemoglobin"], 3, labels=["low", "middle", "high"])
dff["SIGHT"] = dff.sight_left + dff.sight_right
dff["HEAR"] = dff.hear_left + dff.hear_right
dff["SMK_CAT"] = np.where(dff["smk_stat_type_cd"] > 1, "high", "normal")
dff["GAMMA_CAT"] = pd.qcut(dff["gamma_gtp"], 5, labels=["very_low","low", "middle", "high", "very_high"])


# For BMI values: https://www.ncbi.nlm.nih.gov/books/NBK551660/figure/article-35266.image.f1/?report=objectonly
dff["BMI"] = dff['weight'] / ((dff['height'] / 100) ** 2)
dff["BMI_Cat"] = np.where(dff.BMI < 18.5, "Underweight", "Normal Weight")
dff["BMI_Cat"] = np.where((dff.BMI >= 18.5) & (dff.BMI < 25), "Normal Weight", dff["BMI_Cat"])
dff["BMI_Cat"] = np.where((dff.BMI >= 25) & (dff.BMI < 30), "Overweight", dff["BMI_Cat"])
dff["BMI_Cat"] = np.where((dff.BMI >= 30) & (dff.BMI < 35), "Obese Class I", dff["BMI_Cat"])
dff["BMI_Cat"] = np.where((dff.BMI >= 35) & (dff.BMI < 40), "Obese Class II", dff["BMI_Cat"])
dff["BMI_Cat"] = np.where(dff.BMI  >= 40, "Obese Class III", dff["BMI_Cat"])
dff["OBESE"] = np.where(dff.BMI  >= 30, 1, 0)

dff["TOTAL_CHOLESTEROL"] = dff["ldl_chole"] + dff["hdl_chole"] + dff["triglyceride"]

# Calculate HbA1c
# measure of diabetes control and can be used to assess a person's risk of complications from diabetes.
dff["HEMOGLOBIN_A1C"] = (dff["hemoglobin"] - 47) / 12.1

# The normal value for urine protein in literature is less than 150 mg/24 hours, so our data set includes hourly value
# Calculate urine albumin
# dff["URINE_ALBUMIN"] = (dff["urine_protein"] * 24 - 30) / 100

# Calculate Kidney Function Index
# A composite score considering serum creatinine and urine protein
dff["KIDNEY_FUNCTION_INDEX"] = (dff["serum_creatinine"] / 1.2) + (dff["urine_protein"] / 6)

# Calculate WHtR
# measure of abdominal fat and can be used to assess a person's risk of chronic diseases such as heart disease, stroke, and type 2 diabetes.
dff["WHT_R"] = dff["waistline"] / dff["height"]

# Calculate BPi
# measure of cardiovascular health and can be used to assess a person's risk of heart disease, stroke, and kidney disease.
dff["BPI"] = dff["sbp"] / dff["dbp"]

# Calculate TC/HDL
# measure of cardiovascular health and can be used to assess a person's risk of heart disease, stroke, and kidney disease.
dff["TC/HDL"] = dff["tot_chole"] / dff["hdl_chole"]

# Calculate TG/HDL
# measure of cardiovascular health and can be used to assess a person's risk of heart disease, stroke, and kidney disease.
dff["TG/HDL"] = dff["triglyceride"] / dff["hdl_chole"]

# Calculate SGOT/ALT
# measure of liver function and can be used to assess a person's risk of liver damage.
dff["SGOT/ALT"] = dff["sgot_ast"] / dff["sgot_alt"]

# Calculate Gamma-GTP/AST
#  measure of liver function and can be used to assess a person's risk of liver damage
dff["GAMMA_GTP/AST"] = dff["gamma_gtp"] / dff["sgot_ast"]

# Average sight and hear health
dff["AVG_SIGHT"] = (dff['sight_left'] + dff['sight_right']) / 2
dff['AVG_HEAR'] = (dff['hear_left'] + dff['hear_right']) / 2

# Cardiovascular risk assessment
dff['CV_RISK'] = dff['sbp'] / dff['hdl_chole']

# Metabolic Health
dff['METABOLIC_PANEL'] = dff['hemoglobin'] + dff['serum_creatinine'] + dff['sgot_ast'] + dff['sgot_alt'] + dff['gamma_gtp']

# Calculate Cardiovascular Health Index
dff["CARDIOVASCULAR_HEALTH_INDEX"] = (dff["BPI"] + dff["TC/HDL"] + dff["TG/HDL"]) / 3

# Calculate Liver Function Index
# A composite score considering liver enzyme ratios
dff["LIVER_FUNCTION_INDEX"] = (dff["SGOT/ALT"] + dff["GAMMA_GTP/AST"]) / 2

# Calculate Kidney Function Index
# A composite score considering serum creatinine and urine protein
dff["KIDNEY_FUNCTION_INDEX"] = (dff["serum_creatinine"] / 1.2) + (dff["urine_protein"] / 150)


After Feature Engineering, we look again to see if there is any improvement in our model. Since we are adding new variables, we need to call grab_col_names again and update the num_cols and cat_cols lists beforehand.

In [ ]:
cat_cols, num_cols, cat_but_car = grab_col_names(dff)
print(f"\n{colored('Numerical Columns:','blue', attrs=['reverse'])} {num_cols}\n\n\n{colored('Categorical Columns:','magenta', attrs=['reverse'])} {cat_cols}\n\n\n"
        f"{colored('Cardinal Columns:','cyan', attrs=['reverse'])}{cat_but_car}\n")

In [ ]:
cat_cols.remove(TARGET)

In [ ]:
dff[cat_cols].head()

As a reminder, we do not need to apply encoding if we will use PyCaret. For now, we will use One-Hot Encoding.

In [ ]:
ohe_cols = ["SMK_CAT", "BMI_Cat", "AGE_CAT", "HEMOGLOBIN_CAT", "GAMMA_CAT"]

def one_hot_encoder(dataframe, categorical_cols, drop_first=False):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe

dff = one_hot_encoder(dff, ohe_cols, drop_first=True)

Let's create models one more time and check whether any model performance improvement. You can use PyCaret here, as well. But we want to quickly check the model performance, so we will use the classical approach.

In [ ]:
y = dff[TARGET]
X = dff.drop(TARGET, axis=1)
models = [('LR', LogisticRegression(random_state=12345)),
          ('KNN', KNeighborsClassifier()),
          ('CART', DecisionTreeClassifier(random_state=12345)),
          ('RF', RandomForestClassifier(random_state=12345)),
          ('XGB', XGBClassifier(random_state=12345)),
          ("LightGBM", LGBMClassifier(random_state=12345)),
          # ("CatBoost", CatBoostClassifier(verbose=False, random_state=12345))
         ]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=17)
for name, model in models:
    base_model = model.fit(X_train, y_train)
    y_pred = base_model.predict(X_test)
    print(f"########## {name} ##########")
    print(f"Accuracy: {round(accuracy_score(y_pred, y_test), 4)}")
    print(f"Auc: {round(roc_auc_score(y_pred, y_test), 4)}")
    print(f"Recall: {round(recall_score(y_pred, y_test), 4)}")
    print(f"Precision: {round(precision_score(y_pred, y_test), 4)}")
    print(f"F1: {round(f1_score(y_pred, y_test), 4)}")

In [ ]:
s = setup(dff, target = TARGET, session_id = 123,
          train_size=0.8, fold=3, use_gpu=True)
best = compare_models(sort="F1")

In [ ]:
best

In [ ]:
model = create_model(best)

In [ ]:
plot_model(model, plot = 'feature')

In [ ]:
# BONUS - Classical way to Feature Importance
def plot_importance(model, features, num=len(X), save=False):
    feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': features.columns})
    plt.figure(figsize=(10, 10))
    sns.set(font_scale=1)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",
                                                                     ascending=False)[0:num])
    plt.title('Features')
    plt.tight_layout()
    plt.show()
    if save:
        plt.savefig('importances.png')

plot_importance(model, X)

We can saw that our new features affect the model. According to model performance, feature importance, and other ways, feature engineering can be done again and again until reaches the target.

<a id="7"></a> <br>
# 7. Hyperparameter Optimization with Optuna ⚙

Optuna is an automatic hyperparameter optimization software framework, particularly designed for machine learning. It features an imperative, define-by-run style user API. Thanks to our define-by-run API, the code written with Optuna enjoys high modularity, and the user of Optuna can dynamically construct the search spaces for the hyperparameters. For more information, check the documentation.

> **Documentation: [Optuna Documentation](https://optuna.readthedocs.io/en/stable/index.html)**

Despite our best model being found as CatBoost, it is clearly slow when it is compared with others. Let's consider we decided to create a model with LightGBM and we want to optimize the hyperparameters of a LightGBM model.

Optuna expects an objective function. In this function, we write the hyperparameter values and ranges that we want the Optuna to try, and create a study. Optuna will find a way to find the best parameters.

Increasing the number of trials may give Optuna a chance to find better parameters. We will not make the process longer and limit the trial to 30.

In [ ]:
y = dff[TARGET]
X = dff.drop(TARGET, axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=17)

In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

def objective(trial):
    lgbm_params = dict(
    max_depth = trial.suggest_int("max_depth", 3, 20, log=True),
    early_stopping_round = trial.suggest_int("early_stopping_rounds", 10, 40, log=True),
    scale_pos_weight = trial.suggest_float("scale_pos_weight", 2, 15, log=True),
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1, log=True),
    num_iterations=1000,
    random_state=41,
    verbosity=-99)

    lgbm_model = LGBMClassifier(**lgbm_params).fit(X_train, y_train, eval_set=(X_test, y_test),
                                                  eval_metric="f1",verbose=-99)
    y_pred = lgbm_model.predict(X_test)
    score = f1_score(y_true = y_test, y_pred = y_pred)
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

In [ ]:
# Best hyperparameters found by Optuna
study.best_params

The hyperparameters such as random_state, verbosity, etc. that Optuna did not try to find a value do not appear on best_params. We have to add these parameters to the parameter dictionary before creating the final model.

In [ ]:
params = study.best_params
params["num_iterations"] = 1000
params["random_state"] = 41
params["verbosity"] = -99
params

Now, our final parameter set is ready to create a model.

In [ ]:
lgb_final = LGBMClassifier(**params).fit(X_train, y_train, eval_set=(X_test, y_test), eval_metric="f1",
                                         verbose=-1)

In [ ]:
# Predict with final model
lgb_final.predict(X_test)

Let's give a new hyperparameter set that includes much more hyperparameter.

In [ ]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

def objective(trial):
    lgbm_params = dict(
    max_depth = trial.suggest_int("max_depth", 2, 30),
    early_stopping_round = trial.suggest_int("early_stopping_rounds", 2, 300),
    scale_pos_weight = trial.suggest_float("scale_pos_weight", 1, 15),
    learning_rate = trial.suggest_float("learning_rate", 0.0001, 0.2),
    subsample = trial.suggest_float("subsample", 0.5, 1),
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1),
    colsample_bynode = trial.suggest_float("colsample_bynode", 0.5, 1),
    num_iterations= trial.suggest_int("num_iterations", 50, 2500),
    num_leaves = trial.suggest_int("num_leaves", 10, 500),
    n_jobs = -1,
    random_state=41,
    verbosity=-99)

    lgbm_model = LGBMClassifier(**lgbm_params).fit(X_train, y_train, eval_set=(X_test, y_test),
                                                  eval_metric="f1",verbose=-99)
    y_pred = lgbm_model.predict(X_test)
    score = f1_score(y_true = y_test, y_pred = y_pred)
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

In [ ]:
# add constant parameters
params = study.best_params
params["n_jobs"] = -1
params["random_state"] = 41
params["verbosity"] = -99
params

<a id="8"></a> <br>
# 8. Final Model 🏁

In [ ]:
# Final model
lgb_final = LGBMClassifier(**params).fit(X_train, y_train, eval_set=(X_test, y_test), eval_metric="f1",
                                         verbose=-1)

print(f"Final F1 Score: {f1_score(y_test, lgb_final.predict(X_test))}")

Now, our journey starting with a 0.73 F1 score, ends with 0.76. As I mentioned, model performance can be improved with different approaches to feature engineering, using different models, and trying larger hyperparameter sets. For now, we finished the project here.

<center style="font-family:cursive; font-size:18px; color:#1DB954;">Thank's for reading 🥳<br><br><img src="https://media.tenor.com/nE7CE32ElmMAAAAC/leonardo-di-caprio-cheers.gif" width="500"/></center>